# Stage 2 Notebook 33 - Exp2BB Lane-dominant joint (det interference test)

**Why this exists.** Per the user's hypothesis: does the failing detection head (mAP50 ~0.005 across every Exp2 run) corrupt the shared backbone's gradients in a way that hurts lane?

Evidence from Exp2Z: `val_det_loss` decreased from 2.20 -> 1.93 across 15 epochs while `val/det/metric_map50` stayed at ~0.005. **Det is becoming a backbone-pulling parasite** -- generating gradient signal that doesn't translate to detection capability.

Exp2BB introduces a new `loss.lambda_det` knob: when < 1.0, it scales the detection loss before joint combination. Det still gets full supervision through its own losses; only its weight in the joint backbone-gradient computation is reduced.

Single-config-file change vs Exp2Z: `loss.lambda_det: 0.1`. **Det's influence on the shared backbone is 10x weaker.** Lane gets clean gradient.

Reference: this is the standard 'task interference' diagnostic in multi-task learning. If decoded_f1 jumps when det's backbone influence is suppressed, det was actively hurting lane.

**Important**: this is a DIAGNOSTIC, not the final architecture. The joint model idea is preserved (det still trains via its own loss); we're just isolating the backbone gradient flow.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 15-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp28_rmt_gca_mask_lane_dominant_joint_smoke.log
OK exp28_rmt_gca_mask_lane_dominant_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.2414 det_loss=3.1752 grad_cos=-0.4033 lambda_lane=0.0670
  gate_stats={'gate/det_mean': 0.5012180209159851, 'gate/lane_mean': 0.5026939511299133, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short15'
    EPOCHS = 15
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp28_rmt_gca_mask_lane_dominant_joint_short15 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint_short15.tar --epochs 15 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint_short15.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp28_rmt_gca_mask_lane_dominant_joint_short15_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/data

0

## What to watch in Exp2BB training

Reference Exp2Z: decoded_f1=0.0442, oracle_f1=0.101, matched_iou=0.162, val_det_loss=1.93.

Pass criteria at epoch 15:
- **`val/lane/decoded_f1 >= 0.07`**: 1.6x Exp2Z. Confirms det's backbone influence was hurting lane.
- **`val/lane/decoded_oracle_f1 >= 0.13`**: better backbone -> better geometry ceiling.
- **`val/matched_line_iou >= 0.20`**: lane-clean backbone gradients should improve geometry.
- **`val/det_loss`** rises (since det weight is 10x lower) but `val/det/metric_map50` stays at ~0.005 (det wasn't really learning anyway, so we lose nothing).

Failure signals:
- decoded_f1 stays at ~0.044: det wasn't the bottleneck; backbone gradients are already lane-favoring. Pivot to Exp2DD (KD) or Exp2CC (extended).
- val_det_loss jumps and lane DOESN'T improve: detection task starvation also hurts lane somehow (unexpected). Increase lambda_det back to 0.5.